In [1]:
import os
import glob
import numpy as np
from google import genai
from google.genai import types

CHUNK_SIZE = 250
PROJECT = os.environ.get("GOOGLE_CLOUD_PROJECT")

In [16]:
client = genai.Client(
    vertexai=True,
    enterprise=True,
    project=PROJECT,
    location="global"
)
print("Vertex Client loaded")

Vertex Client loaded


In [3]:
def cosine(a, b):
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))

In [4]:
from chunk_compare import chunk

corpus_chunks = []

for filepath in glob.glob("corpus/*.txt"):
    filename = os.path.basename(filepath)
    with open(filepath, 'r', encoding='utf-8') as f:
        text = f.read()
        
    chunks = chunk(text, CHUNK_SIZE)
    
    for c in chunks:
        corpus_chunks.append({
            "source": filename,
            "text": c
        })

print(f"Total chunks created: {len(corpus_chunks)}")

Total chunks created: 271


In [5]:
def embed_document(text):
    r = client.models.embed_content(
        model="gemini-embedding-001",
        contents=text,
        config=types.EmbedContentConfig(task_type="RETRIEVAL_DOCUMENT"),
    )
    return np.array(r.embeddings[0].values)

print("Embedding chunks... (this might take a minute)")
for doc in corpus_chunks:
    doc["embedding"] = embed_document(doc["text"])
    
print("Indexing complete.")

Embedding chunks... (this might take a minute)
Indexing complete.


In [6]:
def retrieve(query, k=3):
    r = client.models.embed_content(
        model="gemini-embedding-001",
        contents=query,
        config=types.EmbedContentConfig(task_type="RETRIEVAL_QUERY"),
    )
    query_vec = np.array(r.embeddings[0].values)
    
    scores = []
    for doc in corpus_chunks:
        score = cosine(query_vec, doc["embedding"])
        scores.append((score, doc))
        
    scores.sort(key=lambda x: x[0], reverse=True)
    
    return [doc for score, doc in scores[:k]]

In [24]:
rag_schema = types.Schema(
    type=types.Type.OBJECT,
    properties={
        "answer_found_in_context": types.Schema(
            type=types.Type.BOOLEAN,
            description="True if the context contains the answer, False if missing."
        ),
        "answer": types.Schema(
            type=types.Type.STRING,
            description="The answer to the user's question, or a refusal if missing."
        ),
        "supporting_quotes": types.Schema(
            type=types.Type.ARRAY.STRING,
            description="The exact quote(s) from the context that proves the answer. Empty if no answer."
        ),
        "source_documents": types.Schema(
            type=types.Type.ARRAY.STRING,
            description="The source filename(s) the quote(s) came from. Empty if no answer."
        ),
    },
    required=["answer_found_in_context", "answer", "supporting_quotes", "source_documents"]
)

def ask(query):
    retrieved = retrieve(query, k=3)
    
    context_str = ""
    for i, doc in enumerate(retrieved):
        context_str += f"\n--- Source: {doc['source']} ---\n{doc['text'].strip()}\n"
        
    prompt = f"""
    You are an assistant answering questions for a Beloit College community member.
    Use ONLY the provided context below. 
    If the context does not contain the answer, set answer_found_in_context to False and refuse to answer.
    
    Context:
    {context_str}
    
    Question: {query}
    """
    
    max_retries = 3
    for attempt in range(max_retries):
        try:
            response = client.models.generate_content(
                model="gemini-3.5-flash-lite",
                contents=prompt,
                config=types.GenerateContentConfig(
                    response_mime_type="application/json",
                    response_schema=rag_schema,
                    temperature=0.0
                )
            )
            return response.text
        except Exception as e:
            error_msg = str(e)
            if "503" in error_msg or "429" in error_msg:
                print(f"   [!] API busy (Attempt {attempt+1}/{max_retries}). Waiting 10s...")
                import time
                time.sleep(10)
            else:
                print(f"   [x] Fatal API Error: {error_msg}")
                return "{}"
                
    return "{}"

In [25]:
eval_set = [
    {
        "question": "What is the minimum number of units a student must be registered for to be considered full-time?",
        "expected_docs": ["catalog-adding-dropping-withdrawing"]
    },
    {
        "question": "Which students are exempt from the requirement to live in College housing during their first six credit terms?",
        "expected_docs": ["handbook-housing-living-on-campus"]
    },
    {
        "question": "Who must approve an emotional support animal before a student brings it to campus?",
        "expected_docs": ["handbook-disability-and-support-animals"]
    },
    {
        "question": "How many hours per week may an hourly student worker work while classes are in session?",
        "expected_docs": ["union-student-workers-agreement"]
    },
    {
        "question": "A student misses two continuous weeks of a course and is administratively dropped. What happens to their course record, and what happens to their financial aid?",
        "expected_docs": ["handbook-class-attendance", "catalog-satisfactory-academic-progress"]
    },
    {
        "question": "A transfer student arrives with 45 semester hours from a community college and three AP exams scored 4 or 5. How much of that counts toward a Beloit degree, and are they required to live on campus?",
        "expected_docs": ["catalog-admission-and-transfer-credit", "handbook-housing-living-on-campus"]
    },
    {
        "question": "A student withdraws from the college during the third week of the semester. How much tuition is refunded, and what happens to their institutional scholarship?",
        "expected_docs": ["catalog-tuition-payment-and-refunds", "catalog-financial-aid-and-r2t4"]
    },
    {
        "question": "Can I still drop this class without it showing up on my transcript?",
        "expected_docs": ["catalog-adding-dropping-withdrawing"]
    },
    {
        "question": "Can I keep my cat in my room if it's an emotional support animal?",
        "expected_docs": ["handbook-disability-and-support-animals"]
    },
    {
        "question": "How much does a student parking permit cost?",
        "expected_docs": None
    },
    {
        "question": "How many total units of earned credit are required to graduate, and how many of those must be completed at Beloit College?",
        "expected_docs": ["catalog-domain-and-unit-requirements"]
    },
    {
        "question": "What is the maximum number of units with any one course prefix that may count toward the 31 units required for graduation?",
        "expected_docs": ["catalog-domain-and-unit-requirements"]
    },
    {
        "question": "How many units of physical education, recreation, and athletics coursework may count toward the 31 units required for graduation?",
        "expected_docs": ["catalog-domain-and-unit-requirements"]
    },
    {
        "question": "How long does a student have to finish an incomplete, and what happens if they don't?",
        "expected_docs": ["catalog-grading-and-incompletes"]
    },
    {
        "question": "What grade point values does Beloit assign to an A−, a B+, and a C−?",
        "expected_docs": ["catalog-grading-and-incompletes"]
    },
    {
        "question": "What cumulative GPA is required to graduate summa cum laude?",
        "expected_docs": ["catalog-student-records-and-honors"]
    },
    {
        "question": "What term GPA puts a student on academic warning rather than academic probation?",
        "expected_docs": ["catalog-academic-standing"]
    },
    {
        "question": "How much does an Acadeum course cost, and how many Beloit units does a 3-semester-credit Acadeum course earn?",
        "expected_docs": ["registrar-transfer-ap-gce-ib-credit"]
    },
    {
        "question": "Can I take an Acadeum course during the spring semester?",
        "expected_docs": ["registrar-transfer-ap-gce-ib-credit"]
    },
    {
        "question": "A student has 24 earned units and wants to take a course at another college next term. What class year are they, and how much credit can they transfer?",
        "expected_docs": ["catalog-adding-dropping-withdrawing", "registrar-transfer-ap-gce-ib-credit"]
    },
    {
        "question": "A student earns a 3.5 GPA this semester but has an incomplete in one course. Can they make the dean's list?",
        "expected_docs": ["catalog-student-records-and-honors", "catalog-grading-and-incompletes"]
    },
    {
        "question": "A first-year student arrives with AP Statistics scored 5 and AP Psychology scored 4. What credit do they get, and does that satisfy the quantitative reasoning requirement?",
        "expected_docs": ["registrar-transfer-ap-gce-ib-credit", "catalog-domain-and-unit-requirements"]
    },
    {
        "question": "A student fails a course in their first semester and ends with a 0.9 cumulative GPA. Can they repeat the course, and what is their academic standing?",
        "expected_docs": ["catalog-grading-and-incompletes", "catalog-academic-standing"]
    },
    {
        "question": "Can a transfer student use courses from their previous college to satisfy Beloit's writing, quantitative reasoning, and intercultural literacy requirements?",
        "expected_docs": ["catalog-writing-quantitative-intercultural", "registrar-transfer-ap-gce-ib-credit"]
    },
    {
        "question": "I bombed a class last year. If I retake it, does the old grade still count against my GPA?",
        "expected_docs": ["catalog-grading-and-incompletes"]
    },
    {
        "question": "Can my parents see my grades?",
        "expected_docs": ["catalog-grading-and-incompletes", "catalog-student-records-and-honors"]
    },
    {
        "question": "I took an online class over the summer at my community college. Will it count?",
        "expected_docs": ["registrar-transfer-ap-gce-ib-credit"]
    },
    {
        "question": "What's the Wi-Fi password for the residence halls?",
        "expected_docs": None
    },
    {
        "question": "What's the average starting salary for Beloit graduates?",
        "expected_docs": None
    },
]

In [33]:
import json, time
import pandas as pd

def norm(s):
    return " ".join((s or "").lower().split())

rows = []
for i, item in enumerate(eval_set):
    retrieved = retrieve(item["question"], k=3)
    retrieved_sources = [d["source"] for d in retrieved]

    result = json.loads(ask(item["question"]))

    quotes = result.get("supporting_quotes", [])
    cited_docs = result.get("source_documents", [])

    cited_chunks = [d["text"] for d in retrieved if d["source"] in cited_docs]
    combined_cited_text = norm(" ".join(cited_chunks))

    returned_quote = int(len(quotes) > 0)
    
    quote_found = int(all(norm(q) in combined_cited_text for q in quotes)) if quotes else 0

    rows.append({
        "q":              item["question"],
        "answerable":     item["expected_docs"] is not None,
        "hit":            int(all(
                              any(expected in retrieved for retrieved in retrieved_sources) 
                              for expected in item["expected_docs"]
                          )) if item["expected_docs"] else None,
        "returned_quote": returned_quote,
        "quote_found":    quote_found,
        "correct":        None,
        "refused":        int(result.get("answer_found_in_context") is False),
    })
    print(f"Question {i+1}: {result.get("answer")}")
    time.sleep(4)

Question 1: To be considered full-time, a student must be registered for at least 3.0 units, unless approval is received from the advisor and the registrar.
Question 2: Exceptions to the requirement to live in College housing during the first six credit terms include married students, military veterans with two years of active duty, students with dependent children, and commuting students who live with their parents or legal guardians within a 30-minute commute. Any other exceptions require prior approval from the Director of Residence Life (DRL).
Question 3: The Director of Learning Enrichment and Disability Services must grant Emotional Support Animal designation before a student brings the animal to campus.
Question 4: Hourly paid student workers may work up to 20 hours per week while classes are in session during the Academic Year.
Question 5: I am sorry, but the provided context does not contain information about what happens to a student's course record or financial aid after the

In [37]:
df = pd.DataFrame(rows)

correctness = [1,1,1,1,0,0,0,1,1,1,1,1,1,1,1,1,0,1,1,0,1,1,0,1,1,0,1,0,0]

ans   = df[df['answerable'] == True]
unans = df[df['answerable'] == False]
df['correct'] = correctness

hit_rate        = f"{ans['hit'].sum()} of {len(ans)}"
quote_fidelity  = f"{df['quote_found'].sum()} of {df['returned_quote'].sum()}"
correct_answers = f"{df['correct'].sum()} of {len(df)}"
correct_refusal = f"{unans['refused'].sum()} of {len(unans)}"
wrong_refusal   = f"{ans['refused'].sum()} of {len(ans)}"

print(f"hit rate:         {hit_rate}")
print(f"quote fidelity:   {quote_fidelity}")
print(f"correctness:      {correct_answers}")
print(f"correct refusals: {correct_refusal}")
print(f"wrong refusals:   {wrong_refusal}")

df.to_csv("analysis/eval_rows.csv", index=False)

hit rate:         18.0 of 26
quote fidelity:   21 of 21
correctness:      20 of 29
correct refusals: 3 of 3
wrong refusals:   5 of 26
